# Neural Network From Scratch

This notebook reviews a small neural network from two angles:

1. what the math is doing during forward propagation and backpropagation
2. how the same pieces map to a PyTorch model, loss, and optimizer

The scratch implementation uses NumPy only. Softmax and cross-entropy are implemented directly in `nn_numpy.py` rather than delegated to a library helper.

## Model Setup

We use a two-layer classifier:

```text
X -> Linear -> ReLU -> Linear -> logits -> softmax cross-entropy
```

For a batch `X` with shape `(n, d)` and first-layer weights `W1` with shape `(d, h)`, the first affine layer is:

```text
Z1 = X @ W1 + b1
A1 = max(Z1, 0)
logits = A1 @ W2 + b2
```

The implementation keeps this orientation because it makes the NumPy formulas read naturally. PyTorch stores linear weights as `(out_features, in_features)`, so the tie-out transposes weights when copying between implementations.

In [ ]:
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if ROOT.name == "nn_from_scratch":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from nn_from_scratch.nn_numpy import MLP, SGD, SoftmaxCrossEntropy, make_moons_like  # noqa: E402

## Forward Pass

The forward pass produces raw class scores called logits. Logits are not probabilities yet. Cross-entropy converts logits to probabilities with softmax internally, then penalizes the probability assigned to the true class.

The scratch implementation computes stable softmax as:

```text
shifted = logits - row_max(logits)
exp_scores = exp(shifted)
probs = exp_scores / row_sum(exp_scores)
```

Subtracting the row max does not change the softmax probabilities, but it prevents large exponentials from overflowing.

In [ ]:
rng = np.random.default_rng(42)
x_batch = rng.normal(size=(5, 3))
y_batch = np.array([0, 2, 1, 2, 0])

model = MLP(input_dim=3, hidden_dim=4, output_dim=3, rng=np.random.default_rng(123), weight_scale=0.2)
loss_fn = SoftmaxCrossEntropy()

logits = model.forward(x_batch)
loss = loss_fn.forward(logits, y_batch)

print("logits shape:", logits.shape)
print("loss:", loss)
print("probabilities:\n", loss_fn.probs)

## Backpropagation

The useful simplification is that softmax plus cross-entropy has a compact gradient with respect to logits:

```text
dlogits = probs
dlogits[true_class] -= 1
dlogits /= batch_size
```

Then each layer applies the chain rule:

```text
dW = X.T @ dZ
db = row_sum(dZ)
dX = dZ @ W.T
```

For ReLU, gradients pass through only where the original pre-activation was positive.

In [ ]:
grad_logits = loss_fn.backward()
model.backward(grad_logits)

for name, grad in [
    ("fc1.grad_weight", model.fc1.grad_weight),
    ("fc1.grad_bias", model.fc1.grad_bias),
    ("fc2.grad_weight", model.fc2.grad_weight),
    ("fc2.grad_bias", model.fc2.grad_bias),
]:
    print(name, grad.shape)
    print(grad)
    print()

## PyTorch Tie-Out

The script `tieout_pytorch.py` creates the same architecture in PyTorch, copies the NumPy weights into it, and compares:

- logits
- loss
- gradients
- weights after one SGD step

If PyTorch is not installed, the script exits with an install hint. The scratch NumPy code and tests still run without PyTorch.

In [ ]:
from nn_from_scratch.tieout_pytorch import run_tieout

run_tieout()

## Train The Scratch Model

This final section trains only the NumPy model on a small two-moons-like synthetic dataset. The goal is not state-of-the-art accuracy. The goal is to watch the same forward, loss, backward, and SGD pieces run repeatedly in a normal training loop.

In [ ]:
x_train, y_train = make_moons_like(n_samples=300, noise=0.12, rng=np.random.default_rng(10))
train_model = MLP(input_dim=2, hidden_dim=16, output_dim=2, rng=np.random.default_rng(11), weight_scale=0.5)
train_loss = SoftmaxCrossEntropy()
optimizer = SGD(train_model.parameters(), lr=0.3)

history = []
for epoch in range(201):
    logits = train_model.forward(x_train)
    loss = train_loss.forward(logits, y_train)
    train_model.backward(train_loss.backward())
    optimizer.step()

    if epoch % 20 == 0:
        preds = train_model.predict(x_train)
        accuracy = (preds == y_train).mean()
        history.append((epoch, loss, accuracy))

for epoch, loss, accuracy in history:
    print(f"epoch={epoch:3d} loss={loss:.4f} accuracy={accuracy:.3f}")

## Review Prompts

After running the notebook, useful checks are:

- Which arrays are cached during the forward pass, and why are they needed for backward?
- Why does PyTorch store `Linear.weight` transposed relative to the NumPy implementation?
- Where does the `1 / batch_size` factor enter the gradient?
- What would change if the loss used a sum instead of a mean?
- Which parts would need to change to add another hidden layer?